# 🧪 معمل التجارب: تحديث فهرس المتجهات وإضافة مستندات جديدة (Lecture 29)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في الحياة العملية، قواعد المعرفة متغيرة ومستمرة في النمو؛ كل يوم بيوصل للشركة تقارير ومستندات جديدة.
هل من المنطقي كل ما يجي ملف جديد نمسح الفهرس ونعيده من الصفر؟ طبعاً لأ!
في هذا المعمل سنتعلم كيفية استخدام `add_documents` لإلحاق المعرفة الجديدة بفهرس `ChromaDB` الموجود فورياً، مع توليد معرفات ثابتة (Deterministic IDs) لتفعيل الـ Upsert ومنع تكرار البيانات نهائياً.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات
**الفكرة ببساطة:**
بنستورد كائن `Document` ومكونات التقطيع والتخزين ونحدد مسار الفهرس الموجود على الهارد داخل `data/databases/chroma_vector_store`.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
docs_dir = data_dir / "chroma_sample_data"
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("✅ Libraries imported successfully!")
print(f"📁 Target Database: {db_dir}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتحديد المسار بنفسك هنا):



### 2. الاتصال بالفهرس القائم وفحص عدد السجلات قبل التحديث
**الفكرة ببساطة:**
بنفتح قاعدة بيانات ChromaDB الموجودة على الهارد ونشوف عدد المتجهات قبل أي تعديل باستخدام خاصية `_collection.count()`.

In [ ]:
# initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect to the existing persistent chroma database
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

# check document count before update
initial_count = vector_store._collection.count()
print(f"✅ Connected to vector store at: {db_dir}")
print(f"📊 Document count BEFORE update: {initial_count}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب الاتصال بقاعدة المتجهات وفحص عددها بنفسك هنا):



### 3. إنشاء المستند الجديد وإرفاق البيانات الوصفية (Metadata)
**الفكرة ببساطة:**
بنجهز مستند جديد عن موضوع (Computer Vision) مكنش موجود في الفهرس قبل كده، ونحط معاه Metadata بتوضح اسم الملف والموضوع، ونحفظ منه نسخة نصية في مجلد البيانات للحفاظ على تزامن الأرشيف.

In [ ]:
# define content for the new document (Computer Vision)
cv_content = (
    "computer vision is a field of artificial intelligence that trains computers to interpret "
    "and understand the visual world. Using digital images from cameras and videos and deep learning models, "
    "machines can accurately identify and classify objects, and then react to what they see. "
    "Key applications include image classification, object detection, and visual recognition."
)

# create new Document instance with metadata
new_doc = Document(
    page_content=cv_content,
    metadata={"source": "docs_3.txt", "topic": "computer_vision"}
)

# save a text copy to the documents directory
os.makedirs(docs_dir, exist_ok=True)
new_doc_path = docs_dir / "docs_3.txt"
new_doc_path.write_text(new_doc.page_content, encoding="utf-8")

print(f"✅ New document prepared with metadata: {new_doc.metadata}")
print(f"💾 Saved copy to: {new_doc_path.name}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء المستند الجديد وإرفاق الـ metadata بنفسك هنا):



### 4. تقطيع المستند الجديد إلى Chunks
**الفكرة ببساطة:**
بنقطع المستند الجديد لقطع صغيرة بنفس مقاس القطع القديمة، والـ Metadata بتنتقل تلقائياً لكل قطعة ناتجة.

In [ ]:
# initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

# split the new document into chunks
new_chunked_docs = text_splitter.split_documents([new_doc])

print(f"✅ New document split into {len(new_chunked_docs)} chunks:\n")
for idx, chunk in enumerate(new_chunked_docs, start=1):
    print(f"  Chunk [{idx}] ({len(chunk.page_content)} chars): {chunk.page_content[:70]}...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تقطيع المستند الجديد بنفسك هنا):



### 5. إضافة القطع وتفادي التكرار بـ Deterministic IDs
**الفكرة ببساطة:**
لو استخدمت `add_documents` بدون IDs، كل ما هتشغل الخلية هينشئ متجهات مكررة جديدة!
عشان كده بننشئ IDs ثابتة مبنية على اسم الملف ورقم القطعة (مثل `docs_3.txt_chunk_0`)، وده بيخلي Chroma يعمل Upsert (تحديث لو موجود، وإضافة لو جديد).

In [ ]:
# generate deterministic IDs to prevent duplicates upon re-running
ids = [f"{chunk.metadata.get('source', 'doc')}_chunk_{i}" for i, chunk in enumerate(new_chunked_docs)]

# add new chunks directly into the existing vector store
added_ids = vector_store.add_documents(new_chunked_docs, ids=ids)

# check updated count
updated_count = vector_store._collection.count()
print(f"✅ Successfully processed {len(added_ids)} vectors with deterministic IDs!")
print(f"🔑 Assigned IDs: {added_ids}")
print(f"📊 Count BEFORE: {initial_count} | Count AFTER: {updated_count}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب توليد الـ IDs وإضافة المستندات بـ add_documents بنفسك هنا):



### 6. التحقق من المعرفة الجديدة بمسار LCEL RAG كامل
**الفكرة ببساطة:**
بنسأل عن الموضوع الجديد اللي لسه ضايفينه (Computer Vision) ونتأكد إن الموديل قادر يسترجعه ويجاوب منه فوراً.

In [ ]:
# helper function to format context chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# prompt & llm
template = """You are a helpful AI assistant. Answer the question using ONLY the provided context.
If you don't know the answer, say "I don't know."
Keep the answer concise and well-structured.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

# construct lcel chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# test query on the newly added document
query = "What is computer vision and what does it train computers to do?"
answer = rag_chain.invoke(query)

print(f"❓ Question: {query}\n")
print(f"💬 Answer:\n{answer}\n")
print("=" * 60)
print("📚 Retrieved Chunks Verification:")
retrieved = retriever.invoke(query)
for idx, doc in enumerate(retrieved, start=1):
    src = doc.metadata.get("source", "unknown")
    topic = doc.metadata.get("topic", "unknown")
    print(f"  [{idx}] (Source: {src} | Topic: {topic}): {doc.page_content[:70]}...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب الاستعلام عن الموضوع الجديد والتحقق من المصادر بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 29):
1. **`add_documents`:** الطريقة القياسية لإلحاق معرفة جديدة بالقاعدة دون إعادة بنائها.
2. **Deterministic IDs:** ميزة احترافية تضمن خاصية الـ Upsert وتمنع تكرار البيانات عند إعادة التشغيل.
3. **التكامل اللحظي:** المستندات الجديدة تتوفر فوراً للـ Retriever ومسار الـ RAG دون الحاجة لأي إعادة تشغيل.